In [19]:
import sys
import os

# Assumes the notebook is in src/distill.
# Go up two levels to the project root.
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))

if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"Added '{project_root}' to sys.path")

# Now you can import from src.model
# For example:
# from src.model.your_file import your_function

Added '/home/brimmann/works/xRAG' to sys.path


In [20]:
from huggingface_hub import notebook_login

notebook_login()

In [3]:
import argparse

In [21]:
def create_prompt_with_mistral_chat_format(messages,tokenizer,*args,**kwargs):
    # return tokenizer.apply_chat_template(messages,tokenize=False,add_special_tokens=False)
    formatted_text = ""
    for message in messages:
        if message['role'] == 'user':
            formatted_text += "[INST] " + message['content'] + " [/INST]"
        elif message['role'] == 'assistant':
            formatted_text += message['content'] + tokenizer.eos_token
        else:
            raise ValueError(
                "Mistral chat template only supports 'user' and 'assistant' roles. Invalid role: {}.".format(message["role"])
                )
    # formatted_text += " The answer is:"
    return formatted_text

In [49]:
def parse_args():
    args = argparse.Namespace(
        # --- Set your desired default values here ---
        retrieval_prefix='colbertv2',
        tf_idf_topk=0,
        base_model=None,  # e.g., 'path/to/base_model'
        use_rag=True,  # This will be set to True if retriever_name_or_path is provided
        enable_progress_bar=True,
        data='triviaqa',  # e.g., 'nq_open', 'hotpotqa', 'triviaqa', 'webqa', 'truthfulqa', 'factkg'
        model_name_or_path='google/gemma-2-2b-it', # 'Hannibal046/xrag-7b', # 'google/gemma-2-2b-it',  e.g., 'path/to/your/model'
        eval_metrics=None,  # This is set based on the 'data' argument below
        n_shot=0,
        retriever_name_or_path='Salesforce/SFR-Embedding-Mistral',#'Salesforce/SFR-Embedding-Mistral',  # e.g., 'colbertv2/colbertv2.0'
        retrieval_topk=[1],
        retrieval_embed_length=0,
        max_test_samples=2,  # e.g., 100 for debugging
        save_dir='./outputs',  # e.g., 'path/to/save/results'
        eval_batch_size=2,
        chat_format='mistral',
    )

    ## post-process
    if args.data in ['nq_open','hotpotqa','triviaqa','webqa']:
        args.task_type = 'open_qa'
        args.eval_metrics = 'substring_match'
    elif args.data in ['truthfulqa']:
        args.task_type = 'open_qa'
        args.eval_metrics = 'truthfulqa_f1_rl'
    elif args.data in ['factkg']:
        args.task_type = 'fact_checking'
        args.eval_metrics = 'fact_checking_acc'
    
    args.retrieval_topk = [x-1 for x in args.retrieval_topk] ## rank starts from 1
    
    if args.chat_format is not None:
        args.chat_format = eval(f"create_prompt_with_{args.chat_format}_chat_format")    
    
    if args.retriever_name_or_path is not None:
        args.use_rag = True

    return args

In [50]:
args = parse_args()

In [51]:
args.max_test_samples

2

In [52]:
args.model_name_or_path

'google/gemma-2-2b-it'

In [27]:
from transformers import (
    AutoTokenizer
)

In [28]:
tokenizer = AutoTokenizer.from_pretrained(
    args.model_name_or_path,
    padding_side = 'left',
    add_eos_token=False, ## import to include this!
    use_fast=False,
)

'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /google/gemma-2-2b-it/resolve/main/tokenizer_config.json (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x79c856b86670>: Failed to resolve \'huggingface.co\' ([Errno -3] Temporary failure in name resolution)"))'), '(Request ID: d3240915-901e-4172-a740-fc960311df0d)')' thrown while requesting HEAD https://huggingface.co/google/gemma-2-2b-it/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].


In [29]:
"<xRAG>" in tokenizer.get_vocab()

False

In [30]:
if tokenizer.pad_token:
    pass
elif tokenizer.unk_token:
    tokenizer.pad_token_id = tokenizer.unk_token_id
elif tokenizer.eos_token:
    tokenizer.pad_token_id = tokenizer.eos_token_id

In [31]:
import torch

In [32]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
retrieval_embed_length = 0
retriever,retriever_tokenizer = None,None

In [33]:
from src.model import (
    XMistralForCausalLM,
    XMixtralForCausalLM,
    SFR,
)

In [53]:
print(args.retriever_name_or_path)

Salesforce/SFR-Embedding-Mistral


In [54]:
if args.retriever_name_or_path is not None:
    
    if args.retriever_name_or_path.lower() == 'salesforce/sfr-embedding-mistral':
        retriever = SFR.from_pretrained(args.retriever_name_or_path,torch_dtype = torch.bfloat16)
        retriever_tokenizer = AutoTokenizer.from_pretrained(args.retriever_name_or_path)
    retrieval_embed_length = retriever.get_embed_length()
    retriever_hidden_size = retriever.get_embed_dim()
    retriever.eval()
    retriever = retriever.to(device)

`torch_dtype` is deprecated! Use `dtype` instead!
'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /Salesforce/SFR-Embedding-Mistral/resolve/main/config.json (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x79c5f545db50>: Failed to resolve \'huggingface.co\' ([Errno -3] Temporary failure in name resolution)"))'), '(Request ID: 79f467eb-05f2-4d95-88ae-a86766aa31bb)')' thrown while requesting HEAD https://huggingface.co/Salesforce/SFR-Embedding-Mistral/resolve/main/config.json
Retrying in 1s [Retry 1/5].
'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /Salesforce/SFR-Embedding-Mistral/resolve/main/config.json (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x79c5eeee3610>: Failed to resolve \'huggingface.co\' ([Errno -3] Temporary failure in name resolution)"))'), '(Request ID: 5b6b8da8-ed54-4d7c-b9b4-baac2e414ef9)')' 

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /Salesforce/SFR-Embedding-Mistral/resolve/main/tokenizer_config.json (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x79c5ef288c40>: Failed to resolve \'huggingface.co\' ([Errno -3] Temporary failure in name resolution)"))'), '(Request ID: e736dcbd-1325-4209-8097-15721d8139bd)')' thrown while requesting HEAD https://huggingface.co/Salesforce/SFR-Embedding-Mistral/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].


In [36]:
from src.eval.run_eval import load_dataset

In [37]:
%cd /home/brimmann/works/xRAG

/home/brimmann/works/xRAG


/home/brimmann/works/xRAG/.venv/lib/python3.9/site-packages/IPython/core/magics/osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [38]:
dev_data,test_data = load_dataset(
    args.data,
    args.use_rag,
    args,
)

In [39]:
len(test_data)

11313

In [40]:
if args.max_test_samples is not None:
    test_data = test_data[:args.max_test_samples]

In [41]:
from src.eval.run_eval import prepare_prompts

In [42]:
prompts,backgrounds = prepare_prompts(
    dev_data = dev_data,
    test_data = test_data,
    task_type = args.task_type,
    tokenizer = tokenizer,
    n_shot = args.n_shot,
    use_rag = args.use_rag,
    retrieval_embed_length = retrieval_embed_length,
    chat_format = args.chat_format, 
)

**************************************** show one example ****************************************
[INST] Refer to the background document and answer the questions:

Background: Alvin and the Chipmunks | " Alvin and the Chipmunks, originally David Seville and the Chipmunks or simply The Chipmunks, are an American animated virtual band created by Ross Bagdasarian for a novelty record in 1958. The group consists of three singing animated anthropomorphic chipmunks named Alvin, Simon, and Theodore. They are managed by their human adoptive father, David ""Dave"" Seville. Bagdasarian provided the group's voices sped up to create high-pitched squeaky voices (which wasn't entirely new to him, having worked on ""Witch Doctor"" earned the record two Grammy Awards for engineering). ""The Chipmunk Song"" became a number-one single in the United States. After Bagdasarian died in 1972, the characters’ voices were provided by his son Ross Bagdasarian Jr. and the latter's wife Janice Karman in the sub

In [43]:
backgrounds

[['Alvin and the Chipmunks | " Alvin and the Chipmunks, originally David Seville and the Chipmunks or simply The Chipmunks, are an American animated virtual band created by Ross Bagdasarian for a novelty record in 1958. The group consists of three singing animated anthropomorphic chipmunks named Alvin, Simon, and Theodore. They are managed by their human adoptive father, David ""Dave"" Seville. Bagdasarian provided the group\'s voices sped up to create high-pitched squeaky voices (which wasn\'t entirely new to him, having worked on ""Witch Doctor"" earned the record two Grammy Awards for engineering). ""The Chipmunk Song"" became a number-one single in the United States. After Bagdasarian died in 1972, the characters’ voices were provided by his son Ross Bagdasarian Jr. and the latter\'s wife Janice Karman in the subsequent incarnations of "'],
 ["Jamie Lee Curtis |  Jamie Lee Curtis (born November 22, 1958) is an American actress and writer. She is the recipient of several accolades, 

In [26]:
prompts

['[INST] Refer to the background document and answer the questions:\n\nBackground: Alvin and the Chipmunks | " Alvin and the Chipmunks, originally David Seville and the Chipmunks or simply The Chipmunks, are an American animated virtual band created by Ross Bagdasarian for a novelty record in 1958. The group consists of three singing animated anthropomorphic chipmunks named Alvin, Simon, and Theodore. They are managed by their human adoptive father, David ""Dave"" Seville. Bagdasarian provided the group\'s voices sped up to create high-pitched squeaky voices (which wasn\'t entirely new to him, having worked on ""Witch Doctor"" earned the record two Grammy Awards for engineering). ""The Chipmunk Song"" became a number-one single in the United States. After Bagdasarian died in 1972, the characters’ voices were provided by his son Ross Bagdasarian Jr. and the latter\'s wife Janice Karman in the subsequent incarnations of "\n\nQuestion: Who was the man behind The Chipmunks?? [/INST] The an

In [44]:
from src.eval.run_eval import prepare_retrieval_embeds

In [46]:
retrieval_embeds = None
if retriever is not None:
    # backgrounds List[List[String]]
    num_samples = len(backgrounds)
    original_orders = []
    for idx,background in enumerate(backgrounds):
        original_orders.extend(
            [idx] * len(background)
        )
        
    backgrounds = [x for y in backgrounds for x in y]
    print(f"Preparing document embedding with {args.retriever_name_or_path}...")
    _retrieval_embeds = prepare_retrieval_embeds(
        backgrounds,
        retriever,
        retriever_tokenizer,
    )

    retrieval_embeds = [[] for _ in range(num_samples)]
    assert len(_retrieval_embeds) == len(original_orders)
    for id,embeds in zip(original_orders,_retrieval_embeds):
        retrieval_embeds[id].append(embeds)

    retriever = retriever.to("cpu")

In [29]:
len(retrieval_embeds)

TypeError: object of type 'NoneType' has no len()

In [58]:
avg_prompt_length = tokenizer(prompts,return_length=True).length
avg_prompt_length = sum(avg_prompt_length)/len(avg_prompt_length)

In [15]:
from transformers import (
    MistralForCausalLM,
    AutoModelForCausalLM,
    AutoTokenizer,
    AutoConfig,
    MixtralForCausalLM,
)

In [17]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Assuming args.model_name_or_path = "google/gemma-2-2b-it"

model = AutoModelForCausalLM.from_pretrained(
    args.model_name_or_path,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    device_map='auto',
    offload_folder="./offload"
)


'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /Hannibal046/xrag-7b/resolve/main/config.json (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x79c856be50d0>: Failed to resolve \'huggingface.co\' ([Errno -3] Temporary failure in name resolution)"))'), '(Request ID: af75d8a7-eefe-45a7-8643-2fce3b0b4038)')' thrown while requesting HEAD https://huggingface.co/Hannibal046/xrag-7b/resolve/main/config.json
Retrying in 1s [Retry 1/5].
`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /Hannibal046/xrag-7b/resolve/main/generation_config.json (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x79c855d1c5e0>: Failed to resolve \'huggingface.co\' ([Errno -3] Temporary failure in name resolution)"))'), '(Request ID: d2127778-70f4-49b8-8cea-561b719c5069)')' thrown while requesting HEAD https://huggingface.co/Hannibal046/xrag-7b/resolve/main/generation_config.json
Retrying in 1s [Retry 1/5].
'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /Hannibal046/xrag-7b/resolve/main/generation_config.json (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x79c855d1c640>: Failed to resolve \'huggingface.co\' ([Errno -3] Temporary failure in name resolution)"))'), '(Request ID: 55f0bda7-8c03-41b9-acc6-24594cc607b3)')' thrown while requesting HEAD https://huggingface.co/Hann

In [ ]:
 ## load llm
config = AutoConfig.from_pretrained(args.model_name_or_path)
MODEL_CLASS = eval(config.architectures[0])
model2 = MODEL_CLASS.from_pretrained(
    args.model_name_or_path,
    torch_dtype = torch.bfloat16,
    low_cpu_mem_usage = True,
    device_map='auto',
    offload_folder="./offload"
)

In [18]:
model.eval()

MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(32002, 4096)
    (layers): ModuleList(
      (0-31): 32 x MistralDecoderLayer(
        (self_attn): MistralAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): MistralMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): MistralRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): MistralRMSNorm((4096,), eps=1e-05)
      )
    )
    (norm): MistralRMSNorm((4096,)

In [62]:
model.eval()

Gemma2ForCausalLM(
  (model): Gemma2Model(
    (embed_tokens): Embedding(256000, 2304, padding_idx=0)
    (layers): ModuleList(
      (0-25): 26 x Gemma2DecoderLayer(
        (self_attn): Gemma2Attention(
          (q_proj): Linear(in_features=2304, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2304, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2304, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2304, bias=False)
        )
        (mlp): Gemma2MLP(
          (gate_proj): Linear(in_features=2304, out_features=9216, bias=False)
          (up_proj): Linear(in_features=2304, out_features=9216, bias=False)
          (down_proj): Linear(in_features=9216, out_features=2304, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (post_attention_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (pre_feedforward_layernorm): Gemma2RMSNo

In [63]:
from src.language_modeling.utils import (
    XRAG_TOKEN,
    get_retrieval_embeds,
)

In [64]:
if retriever is not None:
    assert XRAG_TOKEN in tokenizer.get_vocab() 
    model.set_xrag_token_id(tokenizer.convert_tokens_to_ids(XRAG_TOKEN))

In [65]:
from src.eval.run_eval import llm_for_open_generation

In [66]:
if args.task_type in ['open_qa','fact_checking']:
    generated_results = llm_for_open_generation(
        llm = model,
        llm_tokenizer = tokenizer,
        prompts = prompts,
        retrieval_embeds = retrieval_embeds,
        batch_size = args.eval_batch_size,
        enable_progress_bar= args.enable_progress_bar,
    )

  0%|                             | 0/11313 [00:00<?, ?it/s]

Checking update


  0%|                   | 2/11313 [00:00<1:10:40,  2.67it/s]`eos_token_id` should consist of positive integers, but is tensor([5231, 154547], device='cuda:0'). Your generation will not stop until the maximum length is reached. Depending on other flags, it may even crash.


Checking update
Checking update


`eos_token_id` should consist of positive integers, but is tensor([5231, 154547], device='cuda:0'). Your generation will not stop until the maximum length is reached. Depending on other flags, it may even crash.
  0%|                     | 4/11313 [00:00<32:18,  5.83it/s]`eos_token_id` should consist of positive integers, but is tensor([5231, 154547], device='cuda:0'). Your generation will not stop until the maximum length is reached. Depending on other flags, it may even crash.
`eos_token_id` should consist of positive integers, but is tensor([5231, 154547], device='cuda:0'). Your generation will not stop until the maximum length is reached. Depending on other flags, it may even crash.
`eos_token_id` should consist of positive integers, but is tensor([5231, 154547], device='cuda:0'). Your generation will not stop until the maximum length is reached. Depending on other flags, it may even crash.
  0%|                     | 7/11313 [00:01<18:42, 10.08it/s]`eos_token_id` should consist of

...

`eos_token_id` should consist of positive integers, but is tensor([5231, 154547], device='cuda:0'). Your generation will not stop until the maximum length is reached. Depending on other flags, it may even crash.
`eos_token_id` should consist of positive integers, but is tensor([5231, 154547], device='cuda:0'). Your generation will not stop until the maximum length is reached. Depending on other flags, it may even crash.
100%|████████████████▉| 11307/11313 [09:38<00:00, 14.08it/s]`eos_token_id` should consist of positive integers, but is tensor([5231, 154547], device='cuda:0'). Your generation will not stop until the maximum length is reached. Depending on other flags, it may even crash.
`eos_token_id` should consist of positive integers, but is tensor([5231, 154547], device='cuda:0'). Your generation will not stop until the maximum length is reached. Depending on other flags, it may even crash.


Checking update
Checking update
Checking update
Checking update


100%|████████████████▉| 11309/11313 [09:39<00:00, 15.03it/s]`eos_token_id` should consist of positive integers, but is tensor([5231, 154547], device='cuda:0'). Your generation will not stop until the maximum length is reached. Depending on other flags, it may even crash.
`eos_token_id` should consist of positive integers, but is tensor([5231, 154547], device='cuda:0'). Your generation will not stop until the maximum length is reached. Depending on other flags, it may even crash.
100%|████████████████▉| 11311/11313 [09:39<00:00, 14.97it/s]`eos_token_id` should consist of positive integers, but is tensor([5231, 154547], device='cuda:0'). Your generation will not stop until the maximum length is reached. Depending on other flags, it may even crash.
`eos_token_id` should consist of positive integers, but is tensor([5231, 154547], device='cuda:0'). Your generation will not stop until the maximum length is reached. Depending on other flags, it may even crash.


Checking update
Checking update
Checking update
Checking update


100%|█████████████████| 11313/11313 [09:39<00:00, 19.53it/s]


In [1]:
from src.eval.utils import (
    stop_sequences_criteria,
    get_substring_match_score,
    eval_fact_checking,
    eval_truthfulqa,
    keyword_extraction_with_tfidf,
)

ModuleNotFoundError: No module named 'src'

In [39]:
answers = [x['answer'] for x in test_data]
if args.eval_metrics == 'substring_match':
    score,score_per_sample = get_substring_match_score(generated_results,answers)
elif args.eval_metrics == 'fact_checking_acc':
    score,score_per_sample = eval_fact_checking(generated_results,answers)
elif args.eval_metrics == 'truthfulqa_f1_rl':
    f1,rl,f1_scores,rl_scores = eval_truthfulqa(generated_results,answers)
    score = f"{f1}-{rl}"
    score_per_sample = [(f1_score,rl_score) for f1_score,rl_score in zip(f1_scores,rl_scores)]

In [40]:
result_dict =   {
    "dataset":args.data,
    "batch_size":args.eval_batch_size,
    "include_retrieval":args.use_rag,
    "avg_prompt_length":avg_prompt_length,
    "model":args.model_name_or_path,
    f"{args.eval_metrics}":score,
}

In [41]:
import json

In [42]:
if args.retriever_name_or_path is not None:
    result_dict['retriever'] = args.retriever_name_or_path
print(json.dumps(result_dict,indent=4))

{
    "dataset": "triviaqa",
    "batch_size": 1,
    "include_retrieval": true,
    "avg_prompt_length": 206.25,
    "model": "google/gemma-2-2b-it",
    "substring_match": 0.5
}
